In [ ]:
# Importing the necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import optuna

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Loading the dataset
df = pd.read_csv('../PyTorch Essentials/data/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Splitting the dataset
X = df.iloc[:, 1:].values # numpy array
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
X_train = X_train / 255.0
X_test = X_test / 255.0

In [6]:
# Dataset class
class CustomDataset(Dataset):
    def __init__(self, features, labels):

        # Convert to PyTorch tensors
        self.features = torch.tensor(features, dtype=torch.float32) # type must match with the parameters type
        self.labels = torch.tensor(labels, dtype=torch.long) # long dtype labels require for cross entropy loss

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [7]:
# Objects of dataset
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [8]:
# ------------------------------------------------------------
# 1. Dynamic Model: every trial can have a different architecture
# ------------------------------------------------------------
class DynamicMLP(nn.Module):
    def __init__(self, input_dim, output_dim, layer_configs):
        """
        layer_configs : list of dicts, one per hidden layer
            e.g. [{'units': 128, 'dropout': 0.2, 'activation': 'relu'}, ...]
        """
        super().__init__()
        self.blocks = nn.ModuleList()
        in_dim = input_dim

        for cfg in layer_configs:
            out_dim = cfg["units"]
            block = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                self._activation(cfg["activation"]),
                nn.Dropout(cfg["dropout"]),
            )
            self.blocks.append(block)
            in_dim = out_dim

        self.output = nn.Linear(in_dim, output_dim)

    def _activation(self, name):
        return {
            "relu": nn.ReLU(),
            "leaky_relu": nn.LeakyReLU(),
            "elu": nn.ELU(),
            "tanh": nn.Tanh(),
            "gelu": nn.GELU(),
        }[name]

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)  # flatten if image input
        for block in self.blocks:
            x = block(x)
        return self.output(x)

In [9]:
# ------------------------------------------------------------
# 2. Objective function: builds the model + trains + returns metric
# ------------------------------------------------------------
def objective(trial, train_loader, val_loader, input_dim, output_dim, device):
    # -------- Architecture hyperparameters (dynamic) --------
    n_layers = trial.suggest_int("n_layers", 1, 5)

    layer_configs = []
    for i in range(n_layers):
        # Each layer gets its OWN independent hyperparameters
        n_units = trial.suggest_int(f"n_units_l{i}", 16, 256, log=True)
        dropout = trial.suggest_float(f"dropout_l{i}", 0.0, 0.5)
        activation = trial.suggest_categorical(
            f"activation_l{i}", ["relu", "leaky_relu", "elu", "tanh", "gelu"]
        )
        layer_configs.append(
            {"units": n_units, "dropout": dropout, "activation": activation}
        )

    model = DynamicMLP(input_dim, output_dim, layer_configs).to(device)

    # -------- Training hyperparameters --------
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        # momentum = trial.suggest_float("momentum", 0.9, 0.99)
        optimizer = optim.SGD(
            model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay
        )
    else:
        optimizer = optim.RMSprop(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )

    criterion = nn.CrossEntropyLoss()

    # -------- Training loop with pruning --------
    n_epochs = 20  # can also be tuned: trial.suggest_int('epochs', 10, 50)
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        accuracy = correct / total

        # Report intermediate result for pruning
        trial.report(accuracy, epoch)

        # Let Optuna prune unpromising trials early
        if trial.should_prune():
            raise optuna.TrialPruned()

    return accuracy  # Optuna will maximize this

In [12]:
# ------------------------------------------------------------
# 3. Run the study
# ------------------------------------------------------------
# Dummy data stand-in; replace with your Dataset / DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example: 784-dim input (flattened 28x28), 10 classes
X_train = torch.randn(1000, 784)
y_train = torch.randint(0, 10, (1000,))
X_val = torch.randn(200, 784)
y_val = torch.randint(0, 10, (200,))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=200)

# Use a pruner to kill bad trials early (saves hours)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    direction="maximize",
    pruner=pruner,
    sampler=optuna.samplers.TPESampler(),  # Bayesian optimization
)

# Wrap objective so it receives the data loaders
import functools

obj = functools.partial(
    objective,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=784,
    output_dim=10,
    device=device,
)

study.optimize(obj, n_trials=5, timeout=3600)  # 1 hour cap or 50 trials

print("Best trial:")
print(f"  Accuracy: {study.best_trial.value:.4f}")
print(f"  Params:   {study.best_trial.params}")

# Optional: visualize
# optuna.visualization.plot_param_importances(study).show()
# optuna.visualization.plot_optimization_history(study).show()

[I 2026-05-12 11:59:20,394] A new study created in memory with name: no-name-7d9556ac-91c5-455a-ae6e-cd81bf67ca2b
[I 2026-05-12 12:01:01,893] Trial 0 finished with value: 0.8603333333333333 and parameters: {'n_layers': 5, 'n_units_l0': 21, 'dropout_l0': 0.00518311265959609, 'activation_l0': 'elu', 'n_units_l1': 33, 'dropout_l1': 0.2277616740731277, 'activation_l1': 'tanh', 'n_units_l2': 91, 'dropout_l2': 0.15487280483870702, 'activation_l2': 'leaky_relu', 'n_units_l3': 25, 'dropout_l3': 0.18609118798815777, 'activation_l3': 'relu', 'n_units_l4': 145, 'dropout_l4': 0.06929276988362115, 'activation_l4': 'gelu', 'lr': 0.003436904415191966, 'weight_decay': 0.00012938425301911127, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.8603333333333333.
[I 2026-05-12 12:02:18,878] Trial 1 finished with value: 0.8285833333333333 and parameters: {'n_layers': 2, 'n_units_l0': 34, 'dropout_l0': 0.20774801689450256, 'activation_l0': 'relu', 'n_units_l1': 102, 'dropout_l1': 0.4566692072162076, 'ac

Best trial:
  Accuracy: 0.8603
  Params:   {'n_layers': 5, 'n_units_l0': 21, 'dropout_l0': 0.00518311265959609, 'activation_l0': 'elu', 'n_units_l1': 33, 'dropout_l1': 0.2277616740731277, 'activation_l1': 'tanh', 'n_units_l2': 91, 'dropout_l2': 0.15487280483870702, 'activation_l2': 'leaky_relu', 'n_units_l3': 25, 'dropout_l3': 0.18609118798815777, 'activation_l3': 'relu', 'n_units_l4': 145, 'dropout_l4': 0.06929276988362115, 'activation_l4': 'gelu', 'lr': 0.003436904415191966, 'weight_decay': 0.00012938425301911127, 'optimizer': 'RMSprop'}


If you set n_trials=5, Optuna will run your objective function exactly 5 times.<br>
- Trial 1: Picks a set of hyperparameters, trains, and returns a score.<br>
- Trial 2: Uses the result of Trial 1 to pick a different (hopefully better) set of hyperparameters.<br>
... and so on until Trial 5.